# Receding-horizon replanning (task MPC-1)

The rule behind the console suggestion. When a what-if trigger shrinks the time left, re-check the
rest of the plan and suggest the smallest change that still gets the astronaut home with the reserve
intact. The crew decides; the simulated astronaut does not follow the suggestion on its own.

**Triggers** (Simulation inputs, not measurements):
- **Suit leak:** `leak_min` comes off the consumables budget, so `deadline = budget_min − leak_min − reserve_min`.
- **Dust / visibility:** walking time ÷ `speed_factor`, the walk home included; dwells unchanged.
  Wind is modelled this way, not as a force: at Jezero's air density (about 0.018 kg/m³ from the
  sol 1215 MEDA pressure and temperature, our calculation) a 20 m/s wind pushes like a 2–3 m/s breeze on Earth.

**Definitions:** `D` = walking still to do (lead to the current stop + legs between kept stops + leg
home) ÷ `speed_factor` + remaining dwells, where the current stop needs `max(0, d − dwell spent)`;
`margin = deadline − (t + D)`; `reserve_left = reserve_min + margin`.

**Rule, with the reason for each line** (code: `replanning.py`, `decide`):
1. *Re-check every `replan_interval_min` (5 min), on arrival at each stop and when a trigger changes* — often enough to react within one short leg, and at the moments the crew can act.
2. *If `t + D ≤ deadline`: no suggestion* — the 60 min reserve is already held back inside the deadline.
3. *Else shorten all remaining dwells equally, in 1 min steps, from `dwell_min` down to `min_dwell_min` (10), the longest that fits* — keeps every stop and changes the plan least; whole minutes are easy to say and to reproduce.
4. *Else drop one remaining stop, then two, …, trying every combination in plan order; dwells may be shortened too* — fewer stops is a bigger change than shorter ones, so it comes second.
5. *Choose: most stops kept → longest dwell → larger margin → keep the earlier stops* — science first, then time at each stop, then safety; the last step makes the choice unique.
6. *The stop being walked to is never dropped mid-leg* — the planner routes stop to stop, not from an arbitrary point.
7. *If nothing fits: go home now along `cost_to_home` from (x, y), ÷ `speed_factor`* — always available; a negative margin means the reserve is in use.
8. *Never reorder stops* — the plan's order is the crew's; the rule only removes or shortens.
9. *While a storm alert is active: no suggestion* — mission-control's abort already sends the astronaut home.

Outputs: `out/route.json` (live planner, leg table, commit, cross-check), `out/replanning_cases.json`
(the file MPC-2 is tested against), `out/replanning.png`.

In [1]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import replanning as rp

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "notebooks" / "out"
aoi = json.loads((ROOT / "contracts" / "aoi.json").read_text())
P = rp.load_params(ROOT / "model" / "params.json")
data = json.loads((OUT / "route.json").read_text())
R, LEGS = data["route"], data["legs_min"]
print(P)
print("route fetched", data["fetched_at"], "planner", data["planner_commit"][:7], "totals", R["totals"])
print("cross-check:", data["cross_check"])
assert data["cross_check"]["max_abs_diff"] == 0 and data["cross_check"]["same_paths"]

{'budget_min': 360, 'reserve_min': 60, 'dwell_min': 20, 'min_dwell_min': 10}
route fetched 2026-11-14T16:20:19Z planner 156dd4b totals {'distance_m': 2892.11, 'walk_min': 119.65, 'dwell_min': 100.0, 'eva_min': 219.65}
cross-check: {'local_planner_commit': '156dd4bb43af1d64b1fd68ce3a4b7066890314ca', 'rasters': 'cost_2m, dem_4m, slope_2m from the asset host', 'requests': 21, 'max_abs_diff': 0.0, 'same_paths': True, 'totals_local': {'distance_m': 2892.11, 'walk_min': 119.65, 'dwell_min': 100.0, 'eva_min': 219.65}}


## Data checks

`route.json` was fetched once from the live planner (`POST /route`, URL in DECISIONS.md) with
`rp.fetch`; to refetch, run `rp.fetch(url, stops, commit)` and save it again. `cost_to_home_2m.f32`
comes from the asset host into `data/live/` (git-ignored). Stop if its shape does not match
`contracts/aoi.json`.

In [2]:
px = aoi["rasters"]["cost_to_home"]["px_m"]
w, h = int(aoi["size_m"][0] / px), int(aoi["size_m"][1] / px)
path = ROOT / "data" / "live" / aoi["assets_base"] / aoi["rasters"]["cost_to_home"]["file"]
raw = np.fromfile(path, "<f4")
assert raw.size == w * h, f"cost_to_home has {raw.size} cells, aoi.json says {w} x {h}"
cth = raw.reshape(h, w)

def cost_to_home_at(x, y):
    return float(cth[int((aoi["size_m"][1] - y) // px), int(x // px)])

for s in aoi["stops"]:
    print(f"{s['id']:16s} raster {cost_to_home_at(*s['xy']):6.2f}  planner {LEGS[s['id']]['haven']:6.2f} min home")
    assert abs(cost_to_home_at(*s["xy"]) - LEGS[s["id"]]["haven"]) < 0.5
for stop, t in rp.arrivals(R).items():
    print(f"arrive {stop:16s} t = {t:6.2f} min")

mount-washburn   raster  22.64  planner  22.64 min home
tuff-cliff       raster  26.15  planner  26.15 min home
bright-angel     raster  42.80  planner  42.80 min home
cheyava-falls    raster  43.77  planner  43.77 min home
masonic-temple   raster  47.23  planner  47.23 min home
arrive mount-washburn   t =  22.64 min
arrive tuff-cliff       t =  47.45 min
arrive bright-angel     t =  88.16 min
arrive cheyava-falls    t = 115.21 min
arrive masonic-temple   t = 152.42 min


## Scenarios

In [3]:
cases = rp.build_cases(data, aoi, P, cost_to_home_at)
for c in cases:
    e = c["expected"]
    print(f"{c['id']:34s} {e['action']:15s} {str(e['reason']):11s} dwell {e['dwell_min']:2d}  "
          f"dropped {e['dropped']}  margin {e['margin_min']:7.2f}  reserve left {e['reserve_left_min']:6.2f}")

doc = json.loads((OUT / "replanning_cases.json").read_text())
assert [c["expected"] for c in doc["cases"]] == [c["expected"] for c in cases], "cases file is stale: rewrite it"
print("replanning_cases.json matches", len(cases), "cases")

nominal                            none            plan_fits   dwell 20  dropped []  margin   80.35  reserve left 140.35
dust-0.5-mount-washburn            shorten_dwells  None        dwell 16  dropped []  margin    3.34  reserve left  63.34
dust-0.4-mount-washburn            drop_stops      None        dwell 20  dropped ['masonic-temple']  margin    6.51  reserve left  66.51
dust-0.5-mid-leg                   shorten_dwells  None        dwell 19  dropped []  margin    1.49  reserve left  61.49
leak-90-tuff-cliff                 shorten_dwells  None        dwell 17  dropped []  margin    2.35  reserve left  62.35
leak-120-bright-angel              drop_stops      None        dwell 20  dropped ['masonic-temple']  margin    1.02  reserve left  61.02
leak-120-during-dwell              drop_stops      None        dwell 20  dropped ['masonic-temple']  margin    1.02  reserve left  61.02
leak-150-cheyava-falls             go_home_now     None        dwell  0  dropped ['masonic-temple']  marg

## Figure for the project page

Margin above the reserve: the plan as made (80 min), the trigger with no change, and the trigger
with the suggestion followed. Below zero the astronaut eats into the reserve.

In [4]:
def no_change(s):
    d = P["dwell_min"]
    need = rp.walk_time(s["current_stop"], s["remaining"], s["lead_min"], LEGS) / s.get("speed_factor", 1.0) \
        + max(0.0, d - s["dwell_done_min"]) + len(s["remaining"]) * d
    return P["budget_min"] - s.get("leak_min", 0) - P["reserve_min"] - (s["t_min"] + need), s["t_min"] + need

names = {s["id"]: s["name"] for s in aoi["stops"]}
nominal = P["budget_min"] - P["reserve_min"] - R["totals"]["eva_min"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, cid in zip(axes, ["dust-0.5-mount-washburn", "leak-120-bright-angel"]):
    c = next(c for c in cases if c["id"] == cid)
    s, e = c["input"], c["expected"]
    nc, nc_end = no_change(s)
    te = s["t_min"]
    ax.axhline(0, color="#a50026", lw=1)
    ax.text(335, -2, "reserve line (60 min held back)", color="#a50026", fontsize=8, va="top", ha="right")
    ax.plot([0, te], [nominal, nominal], color="#888", lw=2, label="plan as made")
    ax.plot([te, R["totals"]["eva_min"]], [nominal, nominal], color="#888", lw=1.5, ls=":")
    ax.plot([te, te, nc_end], [nominal, nc, nc], color="#d73027", lw=2, ls="--", label="trigger, no change")
    ax.plot([te, te, e["end_min"]], [nominal, e["margin_min"], e["margin_min"]], color="#1a9850", lw=2.5,
            label="trigger, suggestion followed")
    ax.axvline(te, color="#555", lw=0.8, ls=":")
    trig = f"dust: speed x{s['speed_factor']}" if "speed_factor" in s else f"suit leak: -{s['leak_min']:.0f} min"
    what = {"shorten_dwells": f"shorten dwells to {e['dwell_min']} min",
            "drop_stops": "skip " + ", ".join(names[d] for d in e["dropped"]),
            "go_home_now": "go home now", "none": "no change"}[e["action"]]
    ax.set_title(f"{trig} at {names[s['current_stop']]} (t = {te:.0f} min)\nsuggestion: {what}", fontsize=10)
    ax.set_xlabel("elapsed time (min)")
    ax.set_xlim(0, 340)
axes[0].set_ylabel("margin above reserve (min)")
axes[1].legend(loc="center right", fontsize=8, frameon=False)
fig.text(0.01, 0.005, "SIMULATION: triggers are what-if inputs; route from the Ariadne planner on HiRISE-derived slopes.",
         fontsize=7, color="#555")
fig.tight_layout(rect=(0, 0.03, 1, 1))
fig.savefig(OUT / "replanning.png", dpi=150)